Imports + rutas

In [9]:
from pathlib import Path
import pandas as pd

BASE         = Path("../../")
FORECAST_DIR = BASE / "models" / "forecast"
OUTPUT_DIR   = BASE / "dashboard" / "data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Forecast dir:", FORECAST_DIR)
print("Output dir:",   OUTPUT_DIR)

Forecast dir: ..\..\models\forecast
Output dir: ..\..\dashboard\data


Cargar los 3 forecast .parquet

In [10]:
f_H1 = pd.read_parquet(FORECAST_DIR / "forecast_HOTEL_1.parquet")
f_H2 = pd.read_parquet(FORECAST_DIR / "forecast_HOTEL_2.parquet")
f_H3 = pd.read_parquet(FORECAST_DIR / "forecast_HOTEL_3.parquet")
print("Hotel 1:", f_H1.shape)
print("Hotel 2:", f_H2.shape)
print("Hotel 3:", f_H3.shape)

Hotel 1: (2190, 4)
Hotel 2: (2920, 4)
Hotel 3: (2920, 3)


Unificar en un único dataset

In [11]:
def normalize_forecast(df_raw):
    df = df_raw.copy()
    df = df.drop(
        columns=[c for c in df.columns if c.startswith("__index")],
        errors="ignore"
    )
    if "fecha" in df.columns:
        df["fecha"] = (
            pd.to_datetime(df["fecha"], unit="ms")
            if df["fecha"].dtype != "datetime64[ns]"
            else df["fecha"]
        )
        return df
    df = df.reset_index().rename(columns={"index": "fecha"})
    df["fecha"] = pd.to_datetime(df["fecha"], unit="ms")
    return df

f_H1_n = normalize_forecast(f_H1)
f_H2_n = normalize_forecast(f_H2)
f_H3_n = normalize_forecast(f_H3)
f_H1_n.head(), f_H2_n.head(), f_H3_n.head()

(       fecha    y_pred scenario    hotel
 0 2025-10-21  0.589048      BAU  HOTEL_1
 1 2025-10-22  0.589048      BAU  HOTEL_1
 2 2025-10-23  0.589048      BAU  HOTEL_1
 3 2025-10-24  0.589048      BAU  HOTEL_1
 4 2025-10-25  0.590573      BAU  HOTEL_1,
        fecha    y_pred scenario    hotel
 0 2025-07-18  0.483300      BAU  Hotel 2
 1 2025-07-19  0.481106      BAU  Hotel 2
 2 2025-07-20  0.481106      BAU  Hotel 2
 3 2025-07-21  0.483300      BAU  Hotel 2
 4 2025-07-22  0.483300      BAU  Hotel 2,
                     fecha    y_pred scenario    hotel
 0 1970-01-01 00:00:00.000  0.716607      BAU  HOTEL_3
 1 1970-01-01 00:00:00.001  0.716607      BAU  HOTEL_3
 2 1970-01-01 00:00:00.002  0.716607      BAU  HOTEL_3
 3 1970-01-01 00:00:00.003  0.714614      BAU  HOTEL_3
 4 1970-01-01 00:00:00.004  0.714614      BAU  HOTEL_3)

Concatenar

In [12]:
df_forecast = pd.concat([f_H1_n, f_H2_n, f_H3_n], ignore_index=True)
df_forecast = df_forecast.sort_values(["hotel", "fecha"]).reset_index(drop=True)

Añadir atributos temporales para el dashboard

In [13]:
df_forecast["year"]      = df_forecast["fecha"].dt.year
df_forecast["month"]     = df_forecast["fecha"].dt.month
df_forecast["week"]      = df_forecast["fecha"].dt.isocalendar().week
df_forecast["dayofweek"] = df_forecast["fecha"].dt.dayofweek
df_forecast["dayofyear"] = df_forecast["fecha"].dt.dayofyear

Variables de estación

In [14]:
df_forecast["season_winter"] = df_forecast["month"].isin([12, 1, 2]).astype(int)
df_forecast["season_spring"] = df_forecast["month"].isin([3, 4, 5]).astype(int)
df_forecast["season_summer"] = df_forecast["month"].isin([6, 7, 8, 9]).astype(int)
df_forecast["season_autumn"] = df_forecast["month"].isin([10, 11]).astype(int)

Verificación

In [15]:
print("Escenarios:", df_forecast["scenario"].unique())
print("Hoteles:",    df_forecast["hotel"].unique())
df_forecast.sample(5)

Escenarios: ['BAU' 'ESTACIONAL' 'OTA_PUSH_WINTER' 'OTA_CAP_SUMMER' 'MIX_SHIFT_EX'
 'LOW_DEMAND_PROTECTION' 'J_EARLY_PUSH' 'J_VALLEY_MAX' 'J_CAP_PEAK'
 'MIX_DIVERSIFICATION' 'J_SMART_GLOBAL' 'T_EARLY_PUSH' 'J_VALLEY_RECOVERY'
 'T_CAP_LIMITED' 'MIX_REBALANCE' 'J_SMART_VALLEY']
Hoteles: ['HOTEL_1' 'HOTEL_3' 'Hotel 2']


,fecha,y_pred,scenario,hotel,year,month,week,dayofweek,dayofyear,season_winter,season_spring,season_summer,season_autumn
7171,2026-04-01 00:00:00.000,0.799774,MIX_REBALANCE,Hotel 2,2026,4,14,2,91,0,1,0,0
3545,1970-01-01 00:00:01.355,0.746515,J_VALLEY_MAX,HOTEL_3,1970,1,1,3,1,1,0,0,0
6616,2026-01-22 00:00:00.000,0.783002,T_EARLY_PUSH,Hotel 2,2026,1,4,3,22,1,0,0,0
1465,2026-06-22 00:00:00.000,0.268605,ESTACIONAL,HOTEL_1,2026,6,26,0,173,0,0,1,0
4200,1970-01-01 00:00:02.010,0.604385,MIX_DIVERSIFICATION,HOTEL_3,1970,1,1,3,1,1,0,0,0


Export final

In [16]:
out_file = OUTPUT_DIR / "forecast_summary.parquet"
df_forecast.to_parquet(out_file, index=False)
print("✅ forecast_summary.parquet generado correctamente en:", out_file)

✅ forecast_summary.parquet generado correctamente en: ..\..\dashboard\data\forecast_summary.parquet
